# **Feature Engineering and Refined Baseline**
-----
### What we did

- Started from the cleaned audit dataset  
- Applied **light, targeted feature engineering** (no over-engineering):
  - `comorbidity_count`
  - `chronic_disease_count`
  - `shock_index`
  - `hypoxemia_flag`
  - `is_abg_measured`

- Removed noisy / weak features:
  - ABG raw values (`ph`, `fio2`, `pco2`, `pao2`)
  - high-missing + low-impact labs (`albumin`, `bilirubin`)
  - allergy columns

- Reduced redundancy:
  - dropped `acutephysiologyscore` (highly correlated with `apachescore`)

- Kept feature count controlled (~70 raw → ~90 after encoding)

---

### What we observed

- Missingness is structured and expected (especially labs)
- Models rely heavily on:
  - severity scores (APACHE)
  - core vitals + labs
  - overall patient condition

- Feature engineering did **not drastically change performance**, but:
  - slightly improved stability
  - reduced noise
  - made the dataset cleaner and more interpretable

---

### Model results (summary)

- **Random Forest performed best overall**
  - Recall ≈ 0.48  
  - F1 ≈ 0.57  
  - ROC AUC ≈ 0.86  

- **XGBoost performed similarly**
  - Slightly lower recall (~0.45)

Overall performance is consistent with earlier experiments

---

### Key takeaway

- Simpler, cleaner features > adding more features  
- High-missing variables did not add value in this dataset  
- Most predictive signal comes from:
  - patient severity
  - early vitals/labs
  - ICU context

---

In [3]:
# imports

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [5]:
# load in data 

df = pd.read_csv("simple_preprocess.csv")

print("shape:", df.shape)
df.head()

shape: (2520, 77)


,gender,age,ethnicity,admissionheight,admissionweight,unittype,bmi,dialysis,wbc,respiratoryrate,...,sbp_min,dbp_min,map_min,vitals_missing,hr_range,temp_range,numbedscategory,teachingstatus,region,bad_outcome
0,Female,87.0,Caucasian,157.5,NaN,Med-Surg ICU,NaN,NaN,NaN,NaN,...,84.0,44.0,58.0,1,48.0,1.8,<100,f,Midwest,1
1,Female,87.0,Caucasian,157.5,46.5,Med-Surg ICU,18.745276,0.0,10.2,39.0,...,84.0,44.0,58.0,1,44.0,1.8,<100,f,Midwest,0
2,Male,76.0,Caucasian,167.0,77.5,SICU,27.788734,0.0,11.7,60.0,...,53.0,36.0,47.0,1,15.0,1.8,<100,f,Midwest,0
3,Female,34.0,Caucasian,172.7,60.3,Med-Surg ICU,20.217741,0.0,7.9,6.0,...,84.0,44.0,58.0,1,50.0,1.8,<100,f,Midwest,0
4,Male,61.0,Caucasian,177.8,91.7,SICU,29.007201,0.0,21.1,41.0,...,84.0,44.0,58.0,1,36.0,1.8,<100,f,Midwest,0


-------
## light feature engineering

In [9]:
df = df.copy()

# abg (arterial blood gas) measured flag
abg_cols = ["ph", "fio2", "pco2", "pao2"]
existing_abg = [col for col in abg_cols if col in df.columns]

if len(existing_abg) > 0:
    df["is_abg_measured"] = df[existing_abg].notnull().any(axis=1).astype(int)

# comorbidity count
hx_cols = [
    "hx_cardio", "hx_respiratory", "hx_neuro", "hx_cancer",
    "hx_renal", "hx_liver", "hx_endocrine", "hx_immuno", "hx_heme"
]
hx_cols = [col for col in hx_cols if col in df.columns]

if len(hx_cols) > 0:
    df["comorbidity_count"] = df[hx_cols].fillna(0).sum(axis=1)

# chronic disease count
disease_flag_cols = [
    "aids", "hepaticfailure", "lymphoma", "metastaticcancer",
    "leukemia", "immunosuppression", "cirrhosis", "diabetes"
]
disease_flag_cols = [col for col in disease_flag_cols if col in df.columns]

if len(disease_flag_cols) > 0:
    df["chronic_disease_count"] = df[disease_flag_cols].fillna(0).sum(axis=1)

# shock index
if "heartrate" in df.columns and "sbp_min" in df.columns:
    df["shock_index"] = df["heartrate"] / (df["sbp_min"] + 1e-6)

# hypoxemia flag
if "sao2_min" in df.columns:
    df["hypoxemia_flag"] = (df["sao2_min"] < 90).astype(int)

# NOW drop raw ABG values after creating the flag
df = df.drop(columns=existing_abg, errors="ignore")

print("shape after feature engineering:", df.shape)

shape after feature engineering: (2520, 78)


------

In [12]:
# define target and features
target_col = "bad_outcome"

X = df.drop(columns=[target_col])
y = df[target_col]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (2520, 77)
Target shape: (2520,)


In [14]:
# quick check 
missing_pct = df.isnull().mean().sort_values(ascending=False)
missing_pct[missing_pct > 0].head(25)

bilirubin               0.697222
drug_allergy            0.692857
non_drug_allergy        0.692857
albumin                 0.661111
wbc                     0.398016
hematocrit              0.378175
bun                     0.349206
creatinine              0.347619
sodium                  0.345238
glucose                 0.278968
meanbp                  0.148810
respiratoryrate         0.147619
heartrate               0.141667
shock_index             0.141667
numbedscategory         0.137302
dialysis                0.125000
bmi                     0.086905
region                  0.083333
admissionweight         0.078571
admissionheight         0.027381
acutephysiologyscore    0.016270
apachescore             0.016270
ethnicity               0.015476
nettotal                0.003175
age                     0.001587
dtype: float64

note: hmmmm some final decisions before further modeling

### thought process (keeping it simple)

at this point, the goal is to clean up the dataset **without overcomplicating things**

from earlier attempts:
- the model relies more on **overall patient condition** (APACHE, vitals, core labs)
- some features that are clinically meaningful (like albumin, bilirubin) didn’t actually show strong importance in our models

---

### what we decided to drop

- **albumin & bilirubin**
  - very high missing (~65–70%)
  - weak contribution in feature importance
  - signal is likely already captured by APACHE & other labs

- **drug_allergy, non_drug_allergy**
  - messy + high missing
  - did not contribute meaningful signal
  - even if merginging them, merging weak + weak = still weak is my concern

instead of forcing these into the model, we remove them to reduce noise

---

### redundancy check 

need to check:
- apachescore & acutephysiologyscore

these are highly related measures of severity.

if they are strongly correlated:
- we keep **apachescore**
- drop **acutephysiologyscore**

this can help:
- reduce redundancy
- simplify the model
- avoid splitting importance across similar features

---

In [18]:
#check correlation
corr = df[["apachescore", "acutephysiologyscore"]].corr()
print(corr)

                      apachescore  acutephysiologyscore
apachescore              1.000000              0.946173
acutephysiologyscore     0.946173              1.000000


In [20]:
# way to correlated, keep apachescore, drop APS
df = df.drop(columns=["acutephysiologyscore"], errors="ignore")

print("dropped acutephysiologyscore")
print("new shape:", df.shape)

dropped acutephysiologyscore
new shape: (2520, 77)


In [22]:
# drop high-missing/weak features

drop_cols = [
    "albumin",
    "bilirubin",
    "drug_allergy",
    "non_drug_allergy"
]

existing_drop = [col for col in drop_cols if col in df.columns]

df = df.drop(columns=existing_drop)

print("dropped:", existing_drop)
print("new shape:", df.shape)

dropped: ['albumin', 'bilirubin', 'drug_allergy', 'non_drug_allergy']
new shape: (2520, 73)


In [24]:
# before splitting save cleaned & engineered dataset
df.to_csv("cleaned_feature_engineered_v1.csv", index=False)
print("Saved as cleaned_feature_engineered_v1.csv")

Saved as cleaned_feature_engineered_v1.csv


-----
## split train/test

In [27]:
X = df.drop(columns=["bad_outcome"])
y = df["bad_outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [31]:
# quick look at missingness in training data
X_train.isnull().mean().sort_values(ascending=False).head(25)

wbc                0.394841
hematocrit         0.375000
bun                0.343750
creatinine         0.341270
sodium             0.340278
glucose            0.274802
meanbp             0.148313
respiratoryrate    0.146329
shock_index        0.139881
heartrate          0.139881
numbedscategory    0.137401
dialysis           0.123512
bmi                0.089782
region             0.087302
admissionweight    0.082837
admissionheight    0.027778
apachescore        0.016369
ethnicity          0.015873
nettotal           0.003968
age                0.001488
gender             0.001488
map_min            0.000496
dialysistotal      0.000000
outputtotal        0.000000
intaketotal        0.000000
dtype: float64

In [33]:
# numeric cols
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()

# categorical cols
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()

In [35]:
# numeric pipeline
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),  # handles missing
    ("scaler", StandardScaler())
])

# categorical pipeline
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),  # handles missing
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# combine
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

In [37]:
# basic shapes
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (2016, 72)
X_test : (504, 72)
y_train: (2016,)
y_test : (504,)


In [39]:
# class distribution
print("Train class balance:\n", y_train.value_counts(normalize=True))
print("\nTest class balance:\n", y_test.value_counts(normalize=True))

Train class balance:
 bad_outcome
0    0.768849
1    0.231151
Name: proportion, dtype: float64

Test class balance:
 bad_outcome
0    0.769841
1    0.230159
Name: proportion, dtype: float64


In [41]:
# see how many features after encoding
X_train_proc = preprocessor.fit_transform(X_train)
print("Processed train shape:", X_train_proc.shape)

Processed train shape: (2016, 92)


------
## modeling

In [44]:
# random forest

rf_model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        class_weight="balanced",
        random_state=42
    ))
])

# train
rf_model.fit(X_train, y_train)

# predict
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

# evaluate
print("Random Forest Confusion Matrix:")
print(confusion_matrix(y_test, rf_pred))

print("\nRandom Forest Classification Report:")
print(classification_report(y_test, rf_pred))

print("Random Forest ROC AUC:", roc_auc_score(y_test, rf_prob))

Random Forest Confusion Matrix:
[[365  23]
 [ 60  56]]

Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.94      0.90       388
           1       0.71      0.48      0.57       116

    accuracy                           0.84       504
   macro avg       0.78      0.71      0.74       504
weighted avg       0.82      0.84      0.82       504

Random Forest ROC AUC: 0.8578474937788837


In [46]:
# XGB
xgb_model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    ))
])

# train
xgb_model.fit(X_train, y_train)

# predict
xgb_pred = xgb_model.predict(X_test)
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

# evaluate
print("XGBoost Confusion Matrix:")
print(confusion_matrix(y_test, xgb_pred))

print("\nXGBoost Classification Report:")
print(classification_report(y_test, xgb_pred))

print("XGBoost ROC AUC:", roc_auc_score(y_test, xgb_prob))

XGBoost Confusion Matrix:
[[371  17]
 [ 64  52]]

XGBoost Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.96      0.90       388
           1       0.75      0.45      0.56       116

    accuracy                           0.84       504
   macro avg       0.80      0.70      0.73       504
weighted avg       0.83      0.84      0.82       504

XGBoost ROC AUC: 0.852092961251333


In [48]:
# compare both models

rf_report = classification_report(y_test, rf_pred, output_dict=True)
xgb_report = classification_report(y_test, xgb_pred, output_dict=True)

results_compare = pd.DataFrame([
    {
        "model": "Random Forest",
        "precision_1": rf_report["1"]["precision"],
        "recall_1": rf_report["1"]["recall"],
        "f1_1": rf_report["1"]["f1-score"],
        "roc_auc": roc_auc_score(y_test, rf_prob)
    },
    {
        "model": "XGBoost",
        "precision_1": xgb_report["1"]["precision"],
        "recall_1": xgb_report["1"]["recall"],
        "f1_1": xgb_report["1"]["f1-score"],
        "roc_auc": roc_auc_score(y_test, xgb_prob)
    }
])

results_compare.sort_values(by=["recall_1", "f1_1", "roc_auc"], ascending=False)

,model,precision_1,recall_1,f1_1,roc_auc
0,Random Forest,0.708861,0.482759,0.574359,0.857847
1,XGBoost,0.753623,0.448276,0.562162,0.852093
